# 02. Model Training & Hyperparameter Tuning - Credit Risk Evaluator

**Autor:** Cientista de Dados & Economista  
**Objetivo de Negócio:** Desenvolver, validar e otimizar modelos preditivos de concessão de crédito comparando abordagens paramétricas (Regressão Logística - Scorecard Tradicional) e não-paramétricas baseadas em árvores (Random Forest, LightGBM, XGBoost).

---

### Pipeline de Modelagem:
1. Pré-processamento e Engenharia de Features
2. Validação Cruzada Estratificada (5-Fold StratifiedKFold)
3. Benchmark dos 4 Modelos Candidatos
4. Otimização Bayesiana de Hiperparâmetros (Optuna) para o XGBoost
5. Exportação dos Artefatos de Produção


In [ ]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score
import optuna

# Importação dos módulos do projeto
sys.path.insert(0, os.path.abspath('..'))
from src.data_processing import prepare_data, create_preprocessor, get_feature_names

## 1. Preparação e Pré-processamento dos Dados

In [ ]:
data_path = '../data/raw/credit_risk_dataset.csv'
if not os.path.exists(data_path):
    data_path = 'data/raw/credit_risk_dataset.csv'

X_train, X_test, y_train, y_test = prepare_data(data_path, test_size=0.2, random_state=42)
preprocessor = create_preprocessor()

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
feature_names = get_feature_names(preprocessor)

print(f"Dimensões X_train pré-processado: {X_train_proc.shape}")
print(f"Features geradas ({len(feature_names)}): {feature_names[:6]}...")

## 2. Função de Avaliação Bancária (KS & ROC-AUC)

Em conformidade com Basileia II/III, calculamos a estatística de Kolmogorov-Smirnov (KS) em conjunto com a ROC-AUC.

In [ ]:
def compute_credit_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    auc = roc_auc_score(y_true, y_prob)
    
    # Kolmogorov-Smirnov (KS)
    bads = y_prob[y_true == 1]
    goods = y_prob[y_true == 0]
    ks_stat = float(ks_2samp(bads, goods).statistic)
    
    f1 = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    
    return {
        'ROC-AUC': round(auc, 4),
        'KS': round(ks_stat, 4),
        'F1-Score': round(f1, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4)
    }

## 3. Validação Cruzada Estratificada (5 Folds) dos Modelos Candidatos

In [ ]:
scale_pos = float(np.sum(y_train == 0) / np.sum(y_train == 1))

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, C=0.5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced_subsample', random_state=42, n_jobs=-1),
    'LightGBM': LGBMClassifier(n_estimators=250, learning_rate=0.05, max_depth=6, scale_pos_weight=scale_pos, random_state=42, verbosity=-1, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=250, learning_rate=0.05, max_depth=5, scale_pos_weight=scale_pos, eval_metric='logloss', random_state=42, n_jobs=-1)
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
benchmark_results = []

for name, model in models.items():
    auc_list, ks_list = [], []
    for tr_idx, val_idx in skf.split(X_train_proc, y_train.values):
        model.fit(X_train_proc[tr_idx], y_train.values[tr_idx])
        probs = model.predict_proba(X_train_proc[val_idx])[:, 1]
        m = compute_credit_metrics(y_train.values[val_idx], probs)
        auc_list.append(m['ROC-AUC'])
        ks_list.append(m['KS'])
        
    benchmark_results.append({
        'Modelo': name,
        'ROC-AUC Médio': round(np.mean(auc_list), 4),
        'ROC-AUC Desv': round(np.std(auc_list), 4),
        'KS Statistic': round(np.mean(ks_list), 4)
    })

pd.DataFrame(benchmark_results).sort_values(by='ROC-AUC Médio', ascending=False)

## 4. Otimização de Hiperparâmetros via Optuna (XGBoost)

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 350, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 8),
        'subsample': trial.suggest_float('subsample', 0.65, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 0.95),
        'scale_pos_weight': scale_pos,
        'eval_metric': 'logloss',
        'random_state': 42,
        'n_jobs': -1
    }
    
    cv_scores = []
    for tr_idx, val_idx in skf.split(X_train_proc, y_train.values):
        clf = XGBClassifier(**params)
        clf.fit(X_train_proc[tr_idx], y_train.values[tr_idx])
        probs = clf.predict_proba(X_train_proc[val_idx])[:, 1]
        cv_scores.append(roc_auc_score(y_train.values[val_idx], probs))
    return np.mean(cv_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print(f"Melhor ROC-AUC CV: {study.best_value:.4f}")
print("Melhores Hiperparâmetros:", study.best_params)

## 5. Treinamento Final do Modelo Campeão e Persistência

In [ ]:
best_xgb = XGBClassifier(**study.best_params, scale_pos_weight=scale_pos, eval_metric='logloss', random_state=42)
best_xgb.fit(X_train_proc, y_train.values)

os.makedirs('../models', exist_ok=True)
joblib.dump(best_xgb, '../models/best_model.joblib')
joblib.dump(preprocessor, '../models/preprocessor.joblib')
print("Modelos e transformadores salvos com sucesso em models/!")